# 06 — Synthetic Data Generation

Use Claude to generate gold-standard schema.org JSON-LD for pages that *lack* schema markup.
This is the key step that teaches the model to create schema from scratch — our actual target use case.

**Why**: Pages without schema are underrepresented in WDC. The model needs to learn
to generate schema from visual + HTML cues alone, not copy existing markup.

**Cost estimate** (update via `cost_estimate()` below):
- 5,000 examples × ~3K input tokens × ~500 output tokens
- Claude Sonnet ≈ $50-100 total

**Output**: `data/processed/synthetic/` — one JSON file per generated example

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

SYNTH_DIR = Path('../data/processed/synthetic')
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

from src.synthetic_gen import generate_schema, generate_batch, cost_estimate
from src.schema_validator import validate_jsonld

# Verify API key
api_key = os.getenv('ANTHROPIC_API_KEY')
print('Anthropic API key present:', bool(api_key and api_key.startswith('sk-')))

## Cost Estimate Before Running

In [ ]:
# Check cost before committing
for n in [1_000, 5_000, 10_000]:
    est = cost_estimate(n_examples=n, model='claude-sonnet-4-6')
    print(f"{n:,} examples: ${est['total_cost_usd']:.2f} (${est['input_cost_usd']:.2f} input + ${est['output_cost_usd']:.2f} output)")

## Load Pages Without Schema (Generation Targets)

In [ ]:
# Pages without schema: from notebook 02 cross-reference
# CC domains NOT in WDC = sites that need schema generated

HTML_DIR = Path('../data/raw/html')
SCREENSHOT_DIR = Path('../data/screenshots')

# Build list of pages to generate schema for
# These should be pages where we have HTML but NO existing schema
generation_targets = []

for html_file in HTML_DIR.glob('*.html'):
    with open(html_file, 'r', errors='replace') as f:
        html = f.read()
    
    # Skip if HTML already has good schema (use WDC real-world data instead)
    from src.schema_validator import has_quality_schema
    if has_quality_schema(html, min_score=0.4):
        continue
    
    item_id = html_file.stem
    screenshot_path = SCREENSHOT_DIR / f'{item_id}.png'
    
    generation_targets.append({
        'id': item_id,
        'html': html,
        'screenshot_path': str(screenshot_path) if screenshot_path.exists() else None,
    })

print(f'Pages without quality schema (generation targets): {len(generation_targets)}')

## Test Single Generation

In [ ]:
# Test with a hardcoded example before running at scale
test_html = """
<!DOCTYPE html>
<html>
<head><title>Liffey Valley Auto Repairs</title></head>
<body>
  <h1>Liffey Valley Auto Repairs</h1>
  <p>Expert car servicing and repairs in Dublin since 1987.</p>
  <address>Unit 12, Liffey Valley Business Park, Clondalkin, Dublin 22</address>
  <p>Tel: <a href="tel:+35312345678">01 234 5678</a></p>
  <p>Open Monday-Friday 8am-6pm, Saturday 9am-2pm</p>
  <h2>Services</h2>
  <ul>
    <li>NCT Pre-check</li>
    <li>Full service</li>
    <li>Brake replacement</li>
    <li>Tyres</li>
  </ul>
  <p>All makes and models. Free estimates. <a href="/contact">Book online</a></p>
</body>
</html>
"""

result = generate_schema(html=test_html)
print('Generated schema:')
if result:
    try:
        parsed = json.loads(result)
        print(json.dumps(parsed, indent=2))
    except:
        print(result)
    
    validation = validate_jsonld(result)
    print(f'\nValid: {validation["valid"]}')
    print(f'Quality score: {validation["quality_score"]}')
    print(f'Property count: {validation["property_count"]}')
    print(f'Schema types: {validation["schema_types"]}')
else:
    print('Generation failed')

## Batch Generation

In [ ]:
# Run batch generation
# Start with a small batch to verify quality, then scale up

BATCH_SIZE = 100  # Start small; increase once quality is confirmed
batch = generation_targets[:BATCH_SIZE]

print(f'Generating schema for {len(batch)} pages...')
print(f'Estimated cost: ${cost_estimate(len(batch))["total_cost_usd"]:.2f}')
print('\nProceed? Set RUN_BATCH = True to execute')

RUN_BATCH = False  # Set to True to run

if RUN_BATCH:
    results = generate_batch(
        items=batch,
        output_dir=str(SYNTH_DIR),
        model='claude-sonnet-4-6',
        min_quality=0.4,
        requests_per_minute=40,
    )
    
    success = sum(1 for r in results if r['valid'])
    print(f'\nResults: {success}/{len(results)} valid examples generated')

## Review Generated Samples (Manual QA)

Manually review a 5% random sample to check quality before using for training.

In [ ]:
import random

synth_files = list(SYNTH_DIR.glob('*.json'))
print(f'Total synthetic examples: {len(synth_files)}')

if synth_files:
    sample_size = max(1, len(synth_files) // 20)  # 5% sample
    sample_files = random.sample(synth_files, sample_size)
    
    for f in sample_files[:3]:  # Show first 3
        with open(f) as fh:
            rec = json.load(fh)
        print(f'\n--- {rec["id"]} ---')
        print(f'URL: {rec.get("source_url", "N/A")}')
        print(f'Quality: {rec.get("quality_score", 0):.2f}')
        if rec.get('generated_schema'):
            try:
                parsed = json.loads(rec['generated_schema'])
                print(json.dumps(parsed, indent=2)[:500] + '...')
            except:
                print(rec['generated_schema'][:200])

In [ ]:
# Quality distribution of synthetic examples
import matplotlib.pyplot as plt

quality_scores = []
schema_types = []

for f in synth_files:
    with open(f) as fh:
        rec = json.load(fh)
    if rec.get('valid'):
        quality_scores.append(rec.get('quality_score', 0))
        schema_types.extend(rec.get('schema_types', []))

if quality_scores:
    plt.figure(figsize=(10, 4))
    plt.hist(quality_scores, bins=20)
    plt.xlabel('Quality Score')
    plt.ylabel('Count')
    plt.title(f'Synthetic Data Quality Distribution (n={len(quality_scores)})')
    plt.axvline(x=0.4, color='r', linestyle='--', label='Min threshold')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print(f'Mean quality: {sum(quality_scores)/len(quality_scores):.3f}')

print('\nRun notebook 05 again after this to merge synthetic data into training set')
print('Next step: 07_fine_tuning.ipynb')